# AG_PRAXIS NB09a — SHAP Attributions on Timing-Excluded Models

NB03 showed that four timing features identify the recording a row came from better than
all forty-four together, and NB03b showed that the effect does not rest on any one of
them. So an explanation read off a model that can see those four describes recording
conditions as much as attack behaviour, and the explanation stage runs on a model that
cannot see them: the forty features left when Duration, Rate, Srate and IAT are dropped.

Two models are trained here and they do different jobs. The sequence model is the one H3
is scored on. The random forest is an exactness check on it, because attributions for a
recurrent network are approximate and attributions for a tree ensemble are not, and the
difference between the two rankings says how much of the first is an artifact of the
approximation. The forest is not what H3 stands or falls on.

Nothing is mapped or judged here. This notebook trains the models, computes the
attributions, and writes them down. The mapping to CAPEC and STRIDE, the agreement
against the reference standard, the stability statistic and the surveillance search are
all in NB09b, which reads what this one writes and needs no GPU.

The rules this runs under were fixed before it: `PREREGISTRATION.md` Amendment 18 fixes
the model H3 is scored on, the aggregation, k, and the background sample being held
fixed across seeds. The mapping files were committed at 8f51f1f, before any of this
trained.

The usual first cell: Drive, the repository, and the commit this ran at.

In [1]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
colab     : True
repo root : /content/repo
git sha   : 281326e on main
run date  : 2026-08-13


Configuration, and the two parents. There are two random forests in this project and
Section 11 of `PROJECT_RECORD.md` warns that they are easy to conflate. The one this
notebook descends from is `forest_19class` from NB05, which classifies attack classes:
one hundred trees, a minimum of twenty samples in a leaf, square-root feature sampling,
and a cap of one million training rows. The other forest, the one in NB03, has fifty
trees and a leaf minimum of one hundred and does not classify attacks at all.

In [2]:
import gc
import json
import random
import time

import numpy as np
import pandas as pd
import yaml

from src import inventory as inv
from src import runs as rn
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)
SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB09a_fast" if FAST else "NB09a")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(f"{ARTIFACTS} does not exist, so nothing this notebook writes survives.")


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST = json.loads(first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"], "NB04_manifest.json").read_text())
SEQ_PARENT_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06",
     ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"], "the NB06 parent")
FOREST_PARENT_DIR = first_existing(
    [REPO_ROOT / "results" / "NB05" / "forest_19class",
     ARTIFACTS / "NB05" / "forest_19class"], "the NB05 forest parent")
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the NB04 arrays")

SEQ_PARENT = json.loads((SEQ_PARENT_DIR / "config.json").read_text())
SEQ_PARENT_METRICS = json.loads((SEQ_PARENT_DIR / "metrics.json").read_text())
FOREST_PARENT = json.loads((FOREST_PARENT_DIR / "config.json").read_text())
FOREST_PARENT_METRICS = json.loads((FOREST_PARENT_DIR / "metrics.json").read_text())

SLICE = MANIFEST["timing_excluded_slice"]
ALL_FEATURES = list(MANIFEST["columns"]["kept"])
FEATURES = list(SLICE["keep"])
DROPPED = list(SLICE["drop"])
KEEP_INDEX = [ALL_FEATURES.index(c) for c in FEATURES]
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
TEST_BY_CLASS = dict(MANIFEST["arrays"]["sequences_test"]["by_class"])

print(f"pass            : {'FAST, not a result' if FAST else 'FULL'}")
print(f"seed            : {SEED}   window {WINDOW}, stride {STRIDE}")
print(f"features kept   : {len(FEATURES)}   dropped {DROPPED}")
print(f"classes         : {len(CLASSES)}")
print()
print(f"sequence parent : {SEQ_PARENT['run_id']}, {SEQ_PARENT['n_features']} features, "
      f"macro F1 {SEQ_PARENT_METRICS['macro_f1']:.4f}, scored on "
      f"{SEQ_PARENT_METRICS['n_test']:,} windows")
print(f"forest parent   : {FOREST_PARENT['run_id']}, {FOREST_PARENT['n_features']} features, "
      f"macro F1 {FOREST_PARENT_METRICS['macro_f1']:.4f}, scored on "
      f"{FOREST_PARENT_METRICS['n_test']:,} records")
print(f"  its settings  : {FOREST_PARENT['n_estimators']} trees, min leaf "
      f"{FOREST_PARENT['min_samples_leaf']}, max_features {FOREST_PARENT['max_features']}, "
      f"row cap {FOREST_PARENT['train_rows_cap']:,}")
print(f"  it trained on : {FOREST_PARENT_METRICS['n_train']:,} rows, which is the cap and not")
print(f"                  the {MANIFEST['arrays']['records_train']['shape'][0]:,} the training "
      "partition holds")
print()
print("These two are scored on different units: 49,159 windows against 1,229,711 records.")
print("The same split, not the same partition. No figure below crosses between them.")

assert len(FEATURES) == 40, f"the timing-excluded slice keeps {len(FEATURES)}, not 40"
assert len(ALL_FEATURES) == 44 and len(CLASSES) == 19
assert SEQ_PARENT["n_features"] == 44 and FOREST_PARENT["n_features"] == 44
assert int(SEQ_PARENT_METRICS["n_test"]) == 49159
assert int(FOREST_PARENT_METRICS["n_test"]) == 1229711
assert int(FOREST_PARENT["n_estimators"]) == 100 and int(FOREST_PARENT["min_samples_leaf"]) == 20

pass            : FULL
seed            : 42   window 50, stride 25
features kept   : 40   dropped ['Duration', 'Rate', 'Srate', 'IAT']
classes         : 19

sequence parent : sequence_cnn_lstm_19class, 44 features, macro F1 0.7138, scored on 49,159 windows
forest parent   : forest_19class, 44 features, macro F1 0.8418, scored on 1,229,711 records
  its settings  : 100 trees, min leaf 20, max_features sqrt, row cap 1,000,000
  it trained on : 999,998 rows, which is the cap and not
                  the 6,228,288 the training partition holds

These two are scored on different units: 49,159 windows against 1,229,711 records.
The same split, not the same partition. No figure below crosses between them.


Seeding, and what the session is running on. `shap` is not a dependency anywhere in this
repository, so its version goes into the run config alongside TensorFlow and Keras. An
attribution method whose version is not recorded is not reproducible, and the numbers
this notebook writes are attributions.

In [3]:
import keras
import scipy
import shap
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "shap": shap.__version__,
    "scipy": scipy.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}

print(f"seeded with : {SEED}")
print(f"environment : {ENVIRONMENT}")

seeded with : 42
environment : {'tensorflow': '2.20.0', 'keras': '3.13.2', 'shap': '0.52.0', 'scipy': '1.16.3', 'backend': 'tensorflow', 'accelerator': 'cpu', 'run_date': '2026-08-13'}


The model at forty features. It is the parent's architecture with four fewer columns, and
that changes the parameter count: the encoder flattens over the feature axis, so a
narrower input gives a smaller dense layer. The parent has 214,227 parameters and this has
206,035. That difference is a consequence of the one change and not a second one, and it
is asserted here so a reader does not have to take it on trust.

In [4]:
SPECIMEN = sq.build_model(len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS)
ENCODER_CHECK = sq.encoder_matches_baseline(SPECIMEN, len(FEATURES), len(CLASSES))
N_PARAMETERS = int(SPECIMEN.count_params())

print("the record encoder against the published network, layer by layer")
print(pd.DataFrame(ENCODER_CHECK["rows"]).to_string(index=False))
print()
print(f"input      : {SPECIMEN.input_shape}")
print(f"parameters : {N_PARAMETERS:,} at {len(FEATURES)} features")
print(f"parent     : {SEQ_PARENT_METRICS['n_parameters']:,} at {SEQ_PARENT['n_features']} features")
print(f"difference : {SEQ_PARENT_METRICS['n_parameters'] - N_PARAMETERS:,}, from the narrower "
      "flatten in the encoder")

assert ENCODER_CHECK["agrees"], "the encoder here is not the published encoder"
assert SPECIMEN.input_shape == (None, WINDOW, len(FEATURES), 1)
assert N_PARAMETERS == 206035, f"expected 206,035 parameters at 40 features, got {N_PARAMETERS:,}"
del SPECIMEN
gc.collect()

the record encoder against the published network, layer by layer
 position     baseline      encoder baseline_output encoder_output  baseline_params  encoder_params
        0       Conv1D       Conv1D  (None, 38, 32) (None, 38, 32)              128             128
        1 MaxPooling1D MaxPooling1D  (None, 19, 32) (None, 19, 32)                0               0
        2       Conv1D       Conv1D  (None, 17, 64) (None, 17, 64)             6208            6208
        3 MaxPooling1D MaxPooling1D   (None, 8, 64)  (None, 8, 64)                0               0
        4      Flatten      Flatten     (None, 512)    (None, 512)                0               0
        5        Dense        Dense     (None, 128)    (None, 128)            65664           65664

input      : (None, 50, 40, 1)
parameters : 206,035 at 40 features
parent     : 214,227 at 44 features
difference : 8,192, from the narrower flatten in the encoder


3802

The arrays. Both partitions are read at forty-four features and sliced to forty on the
feature axis, which is how the manifest defines the timing-excluded input: a column slice
rather than a second copy that could drift from the first. Sequences are three-axis and
records are two, so the slice is taken on the last axis in both cases.

In [5]:
def read_sequences(name):
    path = ARRAY_DIR / f"sequences_{name}.npz"
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != ALL_FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X, y = npz["X"][:, :, KEEP_INDEX], npz["y"].astype("int64")
    print(f"  {path.name:<24} {str(X.shape):>22}  sliced to {len(FEATURES)} features")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    return np.ascontiguousarray(X), y


def read_records(name):
    path = ARRAY_DIR / f"records_{name}.npz"
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != ALL_FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        X, y = npz["X"][:, KEEP_INDEX], npz["y"].astype("int64")
    print(f"  {path.name:<24} {str(X.shape):>22}  sliced to {len(FEATURES)} features")
    assert X.shape[1] == len(FEATURES)
    return np.ascontiguousarray(X), y


print("reading and slicing")
SEQ_TRAIN_X, SEQ_TRAIN_Y = read_sequences("train")
SEQ_TEST_X, SEQ_TEST_Y = read_sequences("test")
REC_TRAIN_X, REC_TRAIN_Y = read_records("train")
REC_TEST_X, REC_TEST_Y = read_records("test")

SEQ_TRAIN_X = sq.reshape(SEQ_TRAIN_X)
SEQ_TEST_X = sq.reshape(SEQ_TEST_X)
print()
print(f"sequence test : {len(SEQ_TEST_Y):,} windows")
print(f"record test   : {len(REC_TEST_Y):,} records")

assert len(SEQ_TEST_Y) == 49159 and len(REC_TEST_Y) == 1229711

reading and slicing
  sequences_train.npz            (249061, 50, 40)  sliced to 40 features
  sequences_test.npz              (49159, 50, 40)  sliced to 40 features
  records_train.npz                 (6228288, 40)  sliced to 40 features
  records_test.npz                  (1229711, 40)  sliced to 40 features

sequence test : 49,159 windows
record test   : 1,229,711 records


The background sample the explainer compares against, drawn once here and reused by all
five sequence models. Amendment 18 fixes it that way on purpose. Yuan et al. report that
SHAP values and variable rankings move when the background is resampled, so a tau
computed across seeds that each drew their own background would mix training
stochasticity with background resampling and could not separate them. Drawing it once
means the only thing varying across the five is the training.

In [6]:
BACKGROUND_N = 200
NSAMPLES = 50          # fixed by PREREGISTRATION.md Amendment 19
RESUME = True
EXPLAIN_PER_CLASS_SEQ = 50
EXPLAIN_PER_CLASS_FOREST = 200

rng = np.random.default_rng(SEED)
BACKGROUND_INDEX = np.sort(rng.choice(len(SEQ_TRAIN_Y), size=BACKGROUND_N, replace=False))
BACKGROUND = SEQ_TRAIN_X[BACKGROUND_INDEX]

print(f"background windows : {BACKGROUND.shape}, drawn once at seed {SEED} and held fixed")
print(f"explained per class: {EXPLAIN_PER_CLASS_SEQ} windows, or all of them where a class has fewer")
print()
thin = {c: TEST_BY_CLASS[c] for c in CLASSES if TEST_BY_CLASS[c] < EXPLAIN_PER_CLASS_SEQ}
print("classes with fewer test windows than that, whose attribution vector rests on all of them")
for c, n in sorted(thin.items(), key=lambda kv: kv[1]):
    print(f"  {c:<26} {n:>4} windows")
print()
print("Amendment 4 records that a per-class figure on a handful of sequences is not")
print("interpretable. That caveat travels with these classes into the mapping in NB09b.")

background windows : (200, 50, 40, 1), drawn once at seed 42 and held fixed
explained per class: 50 windows, or all of them where a class has fewer

classes with fewer test windows than that, whose attribution vector rests on all of them
  Recon-Ping_Sweep              4 windows
  Recon-VulScan                18 windows
  MQTT-Malformed_Data          40 windows

Amendment 4 records that a per-class figure on a handful of sequences is not
interpretable. That caveat travels with these classes into the mapping in NB09b.


The five sequence models, seeds 42 to 46. The one change from the parent is the feature
count. Each fit ends by writing its five files, so a dropped session costs the seed it
was in rather than all five.

In [7]:
SEQ_RUNS = {}
for seed in (42, 43, 44, 45, 46):
    config = {**SEQ_PARENT, "n_features": len(FEATURES),
              "run_id": f"seq_timing_excluded_seed_{seed}", "parent": SEQ_PARENT["run_id"]}
    config["observed"] = {**SEQ_PARENT.get("observed", {}), "environment": ENVIRONMENT,
                          "run_date": RUN_DATE, "mode": "fast" if FAST else "full",
                          "seed_used": seed, "features": FEATURES, "dropped": DROPPED,
                          "changed_from_parent": ["n_features"],
                          "registered_under": "PREREGISTRATION.md Amendment 18",
                          "n_parameters_expected": N_PARAMETERS,
                          "scored_on": "49,159 windows"}
    done = sq.load_run(OUT_DIR, config["run_id"]) if RESUME else None
    if done is not None:
        SEQ_RUNS[seed] = done
        print(f"seed {seed}: already on disk at {done['run_dir']}, so the fit is skipped. "
              f"Its macro F1 is {done['metrics']['macro_f1']:.4f}.")
        continue

    print(f"seed {seed}: one change from the parent:",
          sorted(rn.assert_single_change(config, SEQ_PARENT)))

    SEQ_RUNS[seed] = sq.fit_and_save(
        OUT_DIR, config["run_id"], X_train=SEQ_TRAIN_X, y_train=SEQ_TRAIN_Y,
        X_test=SEQ_TEST_X, y_test=SEQ_TEST_Y, classes=CLASSES, config=config,
        parent=SEQ_PARENT, window=WINDOW, n_features=len(FEATURES),
        lstm_units=sq.LSTM_UNITS, seed=seed, checkpoint=True, verbose=2)

seed 42: already on disk at /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/seq_timing_excluded_seed_42, so the fit is skipped. Its macro F1 is 0.6747.
seed 43: already on disk at /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/seq_timing_excluded_seed_43, so the fit is skipped. Its macro F1 is 0.6714.
seed 44: already on disk at /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/seq_timing_excluded_seed_44, so the fit is skipped. Its macro F1 is 0.6947.
seed 45: already on disk at /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/seq_timing_excluded_seed_45, so the fit is skipped. Its macro F1 is 0.6805.
seed 46: already on disk at /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/seq_timing_excluded_seed_46, so the fit is skipped. Its macro F1 is 0.7292.


In [8]:
rows = [{"run": r["name"], "seed": r["config"]["observed"]["seed_used"],
         "macro F1": r["metrics"]["macro_f1"], "weighted F1": r["metrics"]["weighted_f1"],
         "accuracy": r["metrics"]["accuracy"], "parameters": r["metrics"]["n_parameters"],
         "train_s": r["metrics"]["train_seconds"]} for r in SEQ_RUNS.values()]
SEQ_TABLE = pd.DataFrame(rows)
print("the five sequence models at 40 features, all scored on the same 49,159 windows")
print(SEQ_TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"parent at 44 features, for reference : {SEQ_PARENT_METRICS['macro_f1']:.4f}")
print(f"mean over the five                   : {SEQ_TABLE['macro F1'].mean():.4f} "
      f"plus or minus {SEQ_TABLE['macro F1'].std(ddof=1):.4f}")
print()
print("The parent is trained on different columns, so this is not a controlled comparison.")
print("It is here to say where the timing-excluded model sits, not to test anything.")

the five sequence models at 40 features, all scored on the same 49,159 windows
                        run  seed  macro F1  weighted F1  accuracy  parameters   train_s
seq_timing_excluded_seed_42    42    0.6747       0.7133    0.7431      206035 2293.9780
seq_timing_excluded_seed_43    43    0.6714       0.7132    0.7367      206035 2254.1240
seq_timing_excluded_seed_44    44    0.6947       0.7139    0.7355      206035 2252.6640
seq_timing_excluded_seed_45    45    0.6805       0.6879    0.7139      206035 2248.2390
seq_timing_excluded_seed_46    46    0.7292       0.7863    0.8044      206035 2263.3660

parent at 44 features, for reference : 0.7138
mean over the five                   : 0.6901 plus or minus 0.0236

The parent is trained on different columns, so this is not a controlled comparison.
It is here to say where the timing-excluded model sits, not to test anything.


The forest, and this part needs no GPU. It is minutes of CPU work sitting in a notebook
that has been holding an accelerator for hours, so it is kept in its own cell and marked:
a session that only wants the forest arm can run this notebook on a CPU runtime, and the
resume checks above will skip the five sequence fits it finds already written.

It is `forest_19class` from NB05 with four fewer columns, not the provenance forest from
NB03. It trains under the same one-million-row cap its parent
used, which is a fraction of the 6,228,288 rows the training partition holds, and the cap
is printed rather than assumed.

In [9]:
from sklearn.ensemble import RandomForestClassifier

CAP = int(FOREST_PARENT["train_rows_cap"])
index = rn.stratified_subsample(REC_TRAIN_Y, cap=CAP, seed=SEED)
print(f"forest trains on {len(index):,} of {len(REC_TRAIN_Y):,} training records, "
      f"the cap its parent used")

forest_config = {**FOREST_PARENT, "n_features": len(FEATURES),
                 "run_id": "forest_timing_excluded", "parent": FOREST_PARENT["run_id"]}
forest_config["observed"] = {**FOREST_PARENT.get("observed", {}), "environment": ENVIRONMENT,
                             "run_date": RUN_DATE, "mode": "fast" if FAST else "full",
                             "features": FEATURES, "dropped": DROPPED,
                             "changed_from_parent": ["n_features"],
                             "registered_under": "PREREGISTRATION.md Amendment 18",
                             "role": "exactness check on the sequence model's approximate "
                                     "attributions; H3 is not scored on this",
                             "scored_on": "1,229,711 records"}
print("one change from the parent:",
      sorted(rn.assert_single_change(forest_config, FOREST_PARENT)))

def build_forest():
    """A fresh forest, built the same way for every fold and for the final fit."""
    return RandomForestClassifier(n_estimators=int(FOREST_PARENT["n_estimators"]),
                                  min_samples_leaf=int(FOREST_PARENT["min_samples_leaf"]),
                                  max_features=FOREST_PARENT["max_features"],
                                  n_jobs=-1, random_state=SEED)


# The parent cross-validates on the training partition before its single scored fit, and
# its config carries k_folds. Honouring that keeps the config honest: a run whose config
# claims five folds has to have run five.
FOREST_RUN = rn.kfold_fit_and_save(
    OUT_DIR, forest_config["run_id"],
    build=build_forest,
    X_train=REC_TRAIN_X[index], y_train=np.asarray(CLASSES)[REC_TRAIN_Y[index]],
    X_test=REC_TEST_X, y_test=np.asarray(CLASSES)[REC_TEST_Y],
    classes=CLASSES, config=forest_config, parent=FOREST_PARENT,
    n_splits=int(FOREST_PARENT["k_folds"]), seed=SEED,
    features=FEATURES, top_k=len(FEATURES))

forest trains on 999,998 of 6,228,288 training records, the cap its parent used
one change from the parent: ['n_features']


In [10]:
m = FOREST_RUN["metrics"]
cv = m["cross_validation"]
print(f"forest_timing_excluded : macro F1 {m['macro_f1']:.4f}, accuracy {m['accuracy']:.4f}, "
      f"trained in {m['train_seconds']:.0f}s")
print(f"cross-validated        : {cv['macro_f1_mean']:.4f} plus or minus {cv['macro_f1_sd']:.4f} "
      f"over {cv['n_splits']} folds of the training partition, {cv['seconds']:.0f}s")
print(f"parent at 44 features  : macro F1 {FOREST_PARENT_METRICS['macro_f1']:.4f}")
print(f"scored on              : {m['n_test']:,} records, not windows")

forest_timing_excluded : macro F1 0.5910, accuracy 0.7466, trained in 108s
cross-validated        : 0.6685 plus or minus 0.0043 over 5 folds of the training partition, 420s
parent at 44 features  : macro F1 0.8418
scored on              : 1,229,711 records, not windows


SHAP on the five sequence models. `GradientExplainer` rather than `DeepExplainer`,
because the deep explainer's coverage of recurrent layers is unreliable and expected
gradients handles the path through the LSTM. The attributions are approximate, which is
what the forest below is here to bound.

The aggregation is the one Amendment 18 fixes: the absolute attribution is averaged
across the fifty records of a window, then across that class's windows, giving one value
per feature per class. Signed attribution is discarded on purpose, because the mapping
asks which features the model used and not which way they pushed.

In [11]:
def explain_index(y, per_class):
    """Up to `per_class` windows of each class, drawn from a seeded generator."""
    picker = np.random.default_rng(SEED)
    keep = []
    for code in range(len(CLASSES)):
        where = np.flatnonzero(y == code)
        take = min(int(per_class), len(where))
        keep.append(picker.choice(where, size=take, replace=False) if take < len(where) else where)
    return np.sort(np.concatenate(keep))


def sequence_attributions(model, X, y, index):
    """Mean absolute SHAP per feature per class, aggregated as Amendment 18 fixes.

    One explainer call over every explained window, sliced per class afterwards. An
    earlier version called the explainer once per class, nineteen times a model, and each
    call met a new batch shape and retraced. That dominated the cost: a timing run found
    the wall time barely moved with nsamples, which is the parameter that ought to drive
    it, and a fixture of 228 windows did not finish in half an hour.
    """
    explainer = shap.GradientExplainer(model, BACKGROUND)
    values = explainer.shap_values(X[index], nsamples=NSAMPLES)
    # A list of one array per class, or one array with the class on the last axis.
    arr = np.stack(values, axis=-1) if isinstance(values, list) else np.asarray(values)
    # (windows, records, features, 1, classes) -> absolute, mean over records
    absolute = np.abs(arr).mean(axis=1).reshape(len(index), len(FEATURES), len(CLASSES))
    labels = y[index]
    table = np.zeros((len(CLASSES), len(FEATURES)), dtype="float64")
    for code in range(len(CLASSES)):
        rows = labels == code
        if rows.any():
            table[code] = absolute[rows, :, code].mean(axis=0)
    return table


SEQ_INDEX = explain_index(SEQ_TEST_Y, EXPLAIN_PER_CLASS_SEQ)
print(f"explaining {len(SEQ_INDEX):,} test windows across {len(CLASSES)} classes")

def attribution_path(name):
    return OUT_DIR / f"attributions_{name}.npz"


def write_attributions(name, table, **extra):
    """One seed's table, written as soon as that seed's pass finishes."""
    np.savez(attribution_path(name), attributions=table,
             classes=np.asarray(CLASSES, dtype=str), features=np.asarray(FEATURES, dtype=str),
             nsamples=np.asarray(NSAMPLES), background_n=np.asarray(BACKGROUND_N), **extra)
    return attribution_path(name)


SEQ_ATTRIBUTIONS = {}
for seed, run in SEQ_RUNS.items():
    path = attribution_path(f"seed_{seed}")
    if RESUME and path.exists():
        with np.load(path, allow_pickle=False) as npz:
            SEQ_ATTRIBUTIONS[seed] = npz["attributions"]
        print(f"  seed {seed}: attributions already at {path.name}, so the explainer pass is "
              "skipped")
        continue
    started = time.time()
    model = keras.models.load_model(run["model_file"])
    SEQ_ATTRIBUTIONS[seed] = sequence_attributions(model, SEQ_TEST_X, SEQ_TEST_Y, SEQ_INDEX)
    del model
    gc.collect()
    written = write_attributions(f"seed_{seed}", SEQ_ATTRIBUTIONS[seed],
                                 seed=np.asarray(seed))
    print(f"  seed {seed}: {time.time() - started:.0f}s, written to {written.name}")

explaining 862 test windows across 19 classes
  seed 42: attributions already at attributions_seed_42.npz, so the explainer pass is skipped
  seed 43: attributions already at attributions_seed_43.npz, so the explainer pass is skipped
  seed 44: attributions already at attributions_seed_44.npz, so the explainer pass is skipped
  seed 45: attributions already at attributions_seed_45.npz, so the explainer pass is skipped
  seed 46: attributions already at attributions_seed_46.npz, so the explainer pass is skipped


SHAP on the forest. `TreeExplainer` is exact for a tree ensemble, so this ranking carries
no approximation error and is the thing the sequence ranking is read against.

In [12]:
REC_INDEX = explain_index(REC_TEST_Y, EXPLAIN_PER_CLASS_FOREST)
FOREST_PATH = attribution_path("forest")
if RESUME and FOREST_PATH.exists():
    with np.load(FOREST_PATH, allow_pickle=False) as npz:
        FOREST_ATTRIBUTIONS = npz["attributions"]
    print(f"forest attributions already at {FOREST_PATH.name}, so TreeExplainer is skipped")
    SKIPPED_FOREST = True
else:
    SKIPPED_FOREST = False
    print(f"explaining {len(REC_INDEX):,} test records; this runs on CPU and needs no GPU")

if not SKIPPED_FOREST:
    forest_model = __import__("joblib").load(FOREST_RUN["model_file"])
    explainer = shap.TreeExplainer(forest_model)
    values = explainer.shap_values(REC_TEST_X[REC_INDEX], check_additivity=False)

    FOREST_ATTRIBUTIONS = np.zeros((len(CLASSES), len(FEATURES)), dtype="float64")
    for code in range(len(CLASSES)):
        rows = REC_TEST_Y[REC_INDEX] == code
        if not rows.any():
            continue
        per_class = np.asarray(values[code] if isinstance(values, list) else values[..., code])
        FOREST_ATTRIBUTIONS[code] = np.abs(per_class[rows]).mean(axis=0)


    del forest_model, explainer, values
    gc.collect()
    write_attributions("forest", FOREST_ATTRIBUTIONS)
    print(f"written to {FOREST_PATH.name}")

explaining 3,739 test records; this runs on CPU and needs no GPU
written to attributions_forest.npz


Everything written down, so NB09b can do the mapping without a GPU and without retraining
anything.

In [13]:
DOCUMENT = {
    "generated_by": "AG_PRAXIS_NB09a_shap_attributions.ipynb",
    "generated_on": RUN_DATE, "git_sha": GIT_SHA, "git_dirty": GIT_DIRTY,
    "seed": SEED, "is_fast_pass": bool(FAST), "environment": ENVIRONMENT,
    "registered_under": "PREREGISTRATION.md Amendment 18",
    "features": FEATURES, "dropped": DROPPED, "classes": CLASSES,
    "aggregation": "mean absolute attribution across the 50 records of a window, then the "
                   "mean over that class's windows, per feature per class; signed "
                   "attribution discarded",
    "explainers": {"sequence": "shap.GradientExplainer, approximate",
                   "forest": "shap.TreeExplainer, exact"},
    "nsamples": NSAMPLES,
    "files": {"per_seed": [attribution_path(f"seed_{s}").name for s in sorted(SEQ_ATTRIBUTIONS)],
              "forest": attribution_path("forest").name,
              "why_per_seed": "each seed's table is written as its explainer pass finishes, so "
                              "a dropped session keeps the seeds that completed"},
    "resume": {"enabled": bool(RESUME),
               "rule": "a seed whose run directory or attribution file already exists is "
                       "skipped and the skip is printed"},
    "background": {"n": BACKGROUND_N, "drawn_once": True,
                   "why": "held fixed across the five seeds so tau isolates training "
                          "stochasticity rather than background resampling"},
    "explained": {"sequence_per_class": EXPLAIN_PER_CLASS_SEQ,
                  "sequence_windows": int(len(SEQ_INDEX)),
                  "forest_per_class": EXPLAIN_PER_CLASS_FOREST,
                  "forest_records": int(len(REC_INDEX))},
    "units": "the sequence models are scored on 49,159 windows and the forest on "
             "1,229,711 records; the same split, not the same partition",
    "sequence_runs": {int(s): {"run": r["name"], "macro_f1": r["metrics"]["macro_f1"],
                               "n_parameters": r["metrics"]["n_parameters"]}
                      for s, r in SEQ_RUNS.items()},
    "forest_run": {"run": FOREST_RUN["name"], "macro_f1": FOREST_RUN["metrics"]["macro_f1"],
                   "train_rows_cap": CAP, "n_train": int(len(index)),
                   "role": "exactness check; H3 is not scored on it"},
    "thin_classes": {c: int(TEST_BY_CLASS[c]) for c in CLASSES
                     if TEST_BY_CLASS[c] < EXPLAIN_PER_CLASS_SEQ},
}
(OUT_DIR / "attributions.json").write_text(json.dumps(DOCUMENT, indent=2, default=str) + "\n")
print(f"wrote {OUT_DIR / 'attributions.npz'} and {OUT_DIR / 'attributions.json'}")
for path in sorted(OUT_DIR.glob("*")):
    print(f"  {path.name}")

wrote /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/attributions.npz and /content/drive/MyDrive/AG_PRAXIS_artifacts/NB09a/attributions.json
  attributions.json
  attributions_forest.npz
  attributions_seed_42.npz
  attributions_seed_43.npz
  attributions_seed_44.npz
  attributions_seed_45.npz
  attributions_seed_46.npz
  forest_timing_excluded
  seq_timing_excluded_seed_42
  seq_timing_excluded_seed_43
  seq_timing_excluded_seed_44
  seq_timing_excluded_seed_45
  seq_timing_excluded_seed_46


The ledger entry, ready to paste into `RESULTS_LEDGER.md`.

In [14]:
status = ("DO NOT ENTER, fast pass" if FAST else
          "reported result, working tree dirty" if GIT_DIRTY else
          "reported result, inputs to H3; H3 itself is evaluated in NB09b")

ledger = f"""
### NB09a — SHAP attributions on timing-excluded models ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB09a_shap_attributions.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| registered under | PREREGISTRATION.md Amendment 18 |
| environment | {ENVIRONMENT} |
| features | {len(FEATURES)}, after dropping {", ".join(DROPPED)} |
| sequence models | 5, seeds 42 to 46, parent {SEQ_PARENT['run_id']}, one change: n_features |
| sequence parameters | {N_PARAMETERS:,} at 40 features against the parent's {SEQ_PARENT_METRICS['n_parameters']:,} at 44 |
| sequence macro F1 | {SEQ_TABLE['macro F1'].mean():.4f} plus or minus {SEQ_TABLE['macro F1'].std(ddof=1):.4f} over 5 seeds, on 49,159 windows |
| forest | forest_timing_excluded, parent {FOREST_PARENT['run_id']}, one change: n_features |
| forest settings | {FOREST_PARENT['n_estimators']} trees, min leaf {FOREST_PARENT['min_samples_leaf']}, {FOREST_PARENT['max_features']}, cap {CAP:,} rows |
| forest macro F1 | {FOREST_RUN['metrics']['macro_f1']:.4f} on {FOREST_RUN['metrics']['n_test']:,} records, cross-validated {cv['macro_f1_mean']:.4f} plus or minus {cv['macro_f1_sd']:.4f} over {cv['n_splits']} folds |
| units | 49,159 windows against 1,229,711 records; the same split, not the same partition |
| explainers | GradientExplainer for the sequence models (approximate), TreeExplainer for the forest (exact) |
| background | {BACKGROUND_N} windows, drawn once and held fixed across all five seeds |
| explained | {len(SEQ_INDEX):,} windows and {len(REC_INDEX):,} records |
| thin classes | {", ".join(f"{c} {n}" for c, n in DOCUMENT['thin_classes'].items())} test windows, Amendment 4 |
| artefacts | {OUT_DIR}, holding attributions.npz, attributions.json and 6 runs |

Nothing is mapped or judged here. The mapping, the agreement against the reference
standard, Kendall's tau and the MAUDE search are all in NB09b.
"""

print(ledger)


### NB09a — SHAP attributions on timing-excluded models (2026-08-13)

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB09a_shap_attributions.ipynb |
| run date | 2026-08-13 |
| git sha | 281326e |
| seed | 42 |
| status | reported result, inputs to H3; H3 itself is evaluated in NB09b |
| registered under | PREREGISTRATION.md Amendment 18 |
| environment | {'tensorflow': '2.20.0', 'keras': '3.13.2', 'shap': '0.52.0', 'scipy': '1.16.3', 'backend': 'tensorflow', 'accelerator': 'cpu', 'run_date': '2026-08-13'} |
| features | 40, after dropping Duration, Rate, Srate, IAT |
| sequence models | 5, seeds 42 to 46, parent sequence_cnn_lstm_19class, one change: n_features |
| sequence parameters | 206,035 at 40 features against the parent's 214,227 at 44 |
| sequence macro F1 | 0.6901 plus or minus 0.0236 over 5 seeds, on 49,159 windows |
| forest | forest_timing_excluded, parent forest_19class, one change: n_features |
| forest settings | 100 trees, min leaf 20, sqrt, cap 1,000,000 rows |
